# Multi-Factor Models and Return Attribution

CAPM explains returns with one factor, the market. Fama and French added size and value, Carhart added momentum, and today
most performance and risk reports decompose a portfolio into exposures to a handful of factors. The mathematics is the
multiple regression of 07/02, run for many assets at once with the pseudoinverse. This notebook builds a four-factor
model on synthetic data, attributes a fund's return and risk to the factors, and uses the model to estimate a large
covariance matrix with far fewer parameters.

## 0. Math link

- [Pseudoinverse](https://github.com/dnlwrthstr/LearningLinearAlgebra/blob/main/hyperskill/05_matrix_decomposition/059_psedoinverse.ipynb):
  $\hat B = X^+Y$ solves all the asset regressions at once, and gives the minimum-norm solution when factors are redundant.
- [Least squares](https://github.com/dnlwrthstr/LearningLinearAlgebra/blob/main/hyperskill/05_matrix_decomposition/0510_least%20squares.ipynb)
  and the normal equations from [07/02](./02_least_squares_and_the_normal_equations.ipynb).
- Euler risk contributions from [05/04](../05_risk_measures/04_risk_contributions_and_euler_allocation.ipynb);
  the problem of estimating a large covariance matrix from [06/04](../06_estimation_error/04_means_are_hard_covariances_are_easier.ipynb).

## 1. Introduction

A "value" fund shows a CAPM alpha of several percent a year. Is the manager skilled, or did the fund simply hold cheap,
small stocks, which have historically earned a premium that anyone can buy through an index product? A multi-factor
regression answers this, and the same model splits the fund's return and risk into pieces a client can understand.

After this notebook you can:

- write a linear factor model and estimate the loadings of many assets in one step with the pseudoinverse;
- compute portfolio factor exposures as weighted averages of asset exposures;
- attribute a portfolio's average excess return to factor premia and alpha;
- attribute its variance to factors and idiosyncratic risk with Euler contributions;
- build a factor-model covariance matrix and compare it with the sample covariance matrix;
- handle redundant factors with the pseudoinverse.

## 2. Theory

### The model

For asset $i$ and month $t$, with $K$ factor returns $f_t \in \mathbb{R}^K$:

$$
r_{i,t} - r_f = \alpha_i + \beta_i^\top f_t + \varepsilon_{i,t}
$$

Typical factors are long-short portfolio returns: **MKT** (market minus risk-free), **SMB** (small minus big), **HML** (high
book-to-market minus low, value), **MOM** (past winners minus past losers). The loadings $\beta_i$ are the asset's exposures.

### Estimating all assets at once

Stack the $n$ months into $X = [\mathbf 1, F] \in \mathbb{R}^{n\times(K+1)}$ and the $N$ assets' excess returns into $Y \in \mathbb{R}^{n\times N}$. Each column of $Y$
is a separate regression with the same $X$, so

$$
\hat B = (X^\top X)^{-1}X^\top Y = X^+Y \in \mathbb{R}^{(K+1)\times N}
$$

The first row holds the alphas, the others the loadings. If two factors are linearly dependent (rank-deficient $X$), $X^\top X$
is singular, but $X^+$ still exists and gives the **minimum-norm** solution: fitted values are unique, the split between
redundant factors is not.

### Portfolios

For portfolio weights $w$, $\beta_p = B^\top w$ and $\alpha_p = \alpha^\top w$: exposures are weighted averages. A portfolio with $B^\top w = 0$ is factor-neutral.

### Return attribution

Averaging the model over the period:

$$
\overline{r_p - r_f} = \hat\alpha_p + \sum_k \hat\beta_{p,k}\,\bar f_k + \bar e_p
$$

with $\bar e_p = 0$ in sample for OLS. Each term $\hat\beta_{p,k}\bar f_k$ is the return earned from exposure to factor $k$; $\hat\alpha_p$ is what
remains. A CAPM alpha that turns into a zero four-factor alpha was a factor premium, not skill.

### Risk attribution

With factor covariance $\Sigma_f$ and idiosyncratic variance $\sigma_{\varepsilon}^2$:

$$
\sigma_p^2 = \beta_p^\top\Sigma_f\beta_p + \sigma_{\varepsilon,p}^2
$$

Euler contributions (05/04) split the systematic part: factor $k$ contributes $\beta_{p,k}(\Sigma_f\beta_p)_k$ to the variance (share of variance) or that
divided by $\sigma_p$ to the volatility. The contributions add up to $\sigma_p^2$ together with $\sigma_{\varepsilon,p}^2$.

### Factor-model covariance matrix

For $N$ assets,

$$
\Sigma = B^\top\Sigma_f B + D, \qquad D = \operatorname{diag}(\sigma_{\varepsilon,1}^2, \dots, \sigma_{\varepsilon,N}^2)
$$

(with $B$ the $K\times N$ loading matrix without the alpha row). It has $NK + K(K+1)/2 + N$ parameters instead of $N(N+1)/2$, is positive definite
whenever $D > 0$, and is far more stable when $N$ is large relative to $n$ (06/04). The price is a modelling assumption: idiosyncratic
returns are uncorrelated across assets.

## 3. Worked Examples

Annual factor premia: MKT 6%, SMB 2%, HML 3%, MOM 5%.

### Example 1: portfolio exposures

Fund A has loadings (MKT, SMB, HML, MOM) = (1.0, 0.6, 0.4, 0.0), fund B (1.1, −0.2, −0.3, 0.4). A 50/50 mix has
$(1.05, 0.2, 0.05, 0.2)$.

### Example 2: return attribution

A fund earned 8.0% per year in excess of the risk-free rate with loadings (1.0, 0.4, 0.3, −0.1):

| source | calculation | return |
|---|---|---|
| market | 1.0 × 6% | 6.0% |
| size | 0.4 × 2% | 0.8% |
| value | 0.3 × 3% | 0.9% |
| momentum | −0.1 × 5% | −0.5% |
| alpha | 8.0% − 7.2% | 0.8% |

### Example 3: risk attribution

Same loadings, factor volatilities 16%, 8%, 9%, 12%, factors uncorrelated for simplicity, idiosyncratic volatility 4%:

$$
\sigma_p^2 = 0.0256 + 0.16\cdot 0.0064 + 0.09\cdot 0.0081 + 0.01\cdot 0.0144 + 0.0016 = 0.02910, \qquad \sigma_p = 17.1\%
$$

Shares of variance: market 88.0%, size 3.5%, value 2.5%, momentum 0.5%, idiosyncratic 5.5%.

### Example 4: parameter count

$N = 500$ stocks, $K = 4$ factors: the sample covariance matrix has $500\cdot 501/2 = 125{,}250$ parameters; the factor model
$500\cdot 4 + 10 + 500 = 2{,}510$.

## 4. Finance Application

### Where it matters

- **Manager evaluation.** Fund alpha is reported against a multi-factor model, so that cheap factor exposure is not paid
  as active fees. "Smart beta" products sell factor exposure explicitly, at lower fees.
- **Risk models.** Commercial equity risk models (Barra, Axioma) are factor models with dozens of industry, country and
  style factors; the covariance matrix of thousands of stocks comes from $B^\top\Sigma_f B + D$.
- **Performance attribution.** Client reports split return into market, style and selection, which is the attribution
  above.
- **Hedging and neutralization.** Market-neutral and factor-neutral portfolios solve $B^\top w = 0$ for the exposures to hedge.

### The usual misreadings

- "Positive CAPM alpha means skill." It may be compensation for size, value or momentum exposure.
- "Zero four-factor alpha means the manager added nothing." It means the return could have been replicated with factor
  portfolios, which may or may not have been available and cheap to the investor.
- "The factor model's covariance matrix is the truth." It assumes idiosyncratic returns are uncorrelated; missing factors
  (industries, countries) leave correlated residuals.

## 5. Python Examples

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

rng = np.random.default_rng(42)
months_per_year = 12

factors = ["MKT", "SMB", "HML", "MOM"]
f_mu = np.array([0.06, 0.02, 0.03, 0.05])             # annual premia, decimal
f_vol = np.array([0.16, 0.08, 0.09, 0.12])            # annual volatilities
f_corr = np.array([[1.00, 0.20, -0.10, -0.15],
                   [0.20, 1.00, 0.05, 0.00],
                   [-0.10, 0.05, 1.00, -0.30],
                   [-0.15, 0.00, -0.30, 1.00]])
Sigma_f = np.outer(f_vol, f_vol) * f_corr             # annual factor covariance

### Simulated factors and 50 stocks

Ten years of monthly factor returns, and 50 stocks with zero alpha, random loadings and 20% idiosyncratic volatility.

In [ ]:
n, N = 120, 50
F = f_mu / 12 + rng.standard_normal((n, 4)) @ np.linalg.cholesky(Sigma_f / 12).T

B_true = np.column_stack([1 + 0.25 * rng.standard_normal(N),      # loadings, shape (4, N) after transpose
                          0.2 + 0.4 * rng.standard_normal(N),
                          0.1 + 0.4 * rng.standard_normal(N),
                          0.2 * rng.standard_normal(N)]).T
idio = np.full(N, 0.20)
Y = F @ B_true + idio / np.sqrt(12) * rng.standard_normal((n, N))   # monthly excess returns

X = np.column_stack([np.ones(n), F])
B_hat_full = np.linalg.pinv(X) @ Y           # (5, N): alphas and loadings for all stocks at once
alpha_hat, B_hat = B_hat_full[0], B_hat_full[1:]
resid = Y - X @ B_hat_full

print("factor averages (annual):", dict(zip(factors, (F.mean(axis=0) * 12).round(4))))
print(f"loading estimation error, RMS by factor: {dict(zip(factors, np.sqrt(((B_hat - B_true)**2).mean(axis=1)).round(3)))}")
print(f"alphas (annual): mean {alpha_hat.mean() * 12:.2%}, std across stocks {alpha_hat.std() * 12:.2%}")
print("same result as lstsq:", np.allclose(B_hat_full, np.linalg.lstsq(X, Y, rcond=None)[0]))

The pseudoinverse and `lstsq` give the same 250 numbers. The market loadings are estimated most precisely (RMS error 0.12),
because the market factor is the most volatile. The realized factor averages over these ten years differ from the
premia: SMB lost 1.2% a year, HML earned 4.5% (06/01: ten years cannot pin down a mean). All stocks have zero alpha, yet the
estimated alphas scatter with a standard deviation of 6% a year across stocks.

### A value fund: CAPM alpha vs four-factor alpha

Fund loadings (1.0, 0.5, 0.8, 0.0), idiosyncratic volatility 3%, fees 0.5% a year (true alpha −0.5%).

In [ ]:
beta_fund = np.array([1.0, 0.5, 0.8, 0.0])
alpha_fund = -0.005
y_fund = alpha_fund / 12 + F @ beta_fund + 0.03 / np.sqrt(12) * rng.standard_normal(n)

def regress(Xm, y):
    b = np.linalg.lstsq(Xm, y, rcond=None)[0]
    e = y - Xm @ b
    se = np.sqrt(e @ e / (len(y) - Xm.shape[1]) * np.diag(np.linalg.inv(Xm.T @ Xm)))
    r2 = 1 - e @ e / ((y - y.mean()) @ (y - y.mean()))
    return b, se, r2

b_capm, se_capm, r2_capm = regress(X[:, :2], y_fund)
b_ff4, se_ff4, r2_ff4 = regress(X, y_fund)
print(f"CAPM      : alpha {b_capm[0] * 12:6.2%} (t = {b_capm[0] / se_capm[0]:5.2f}), beta {b_capm[1]:.2f}, R^2 {r2_capm:.3f}")
print(f"4-factor  : alpha {b_ff4[0] * 12:6.2%} (t = {b_ff4[0] / se_ff4[0]:5.2f}), "
      f"loadings {dict(zip(factors, b_ff4[1:].round(2)))}, R^2 {r2_ff4:.3f}")

Against the market alone the fund shows an alpha of 2.4% per year ($t = 0.85$: ten years are too short to call it significant,
but it is the kind of number a factsheet highlights). Against four factors the alpha is −0.3% (the true value is
the −0.5% fee), and $R^2$ rises from 0.76 to 0.97. The CAPM alpha was the value and size exposure.

### Return and risk attribution of the fund

In [ ]:
f_bar = F.mean(axis=0) * 12
ret_attr = pd.Series(np.r_[b_ff4[1:] * f_bar, b_ff4[0] * 12], index=factors + ["alpha"])
print(f"average excess return {y_fund.mean() * 12:.2%} = sum of parts {ret_attr.sum():.2%}")

Sf_hat = np.cov(F, rowvar=False) * 12
e_fund = y_fund - X @ b_ff4
var_idio = e_fund.var(ddof=5) * 12
contrib_var = b_ff4[1:] * (Sf_hat @ b_ff4[1:])
total_var = b_ff4[1:] @ Sf_hat @ b_ff4[1:] + var_idio
risk_attr = pd.Series(np.r_[contrib_var, var_idio] / total_var, index=factors + ["idiosyncratic"])
print(f"model volatility {np.sqrt(total_var):.2%}, sample volatility {y_fund.std(ddof=1) * np.sqrt(12):.2%}")
order = factors + ["alpha", "idiosyncratic"]
pd.DataFrame({"return contribution (annual)": ret_attr, "share of variance": risk_attr}).reindex(order) \
    .style.format("{:.2%}", na_rep="")

Of the fund's 7.5% average excess return, 5.1 points came from the market, 3.6 from value, −0.6 from size (a bad decade for
small caps in this sample) and −0.3 is alpha. On the risk side the market carries 77% of the variance, value 11%, size 9%;
only 3% is idiosyncratic.

### Factor-model covariance vs sample covariance

With only 60 months of data for the 50 stocks, estimate the covariance matrix both ways, build the minimum-variance
portfolio from each, and evaluate it with the true covariance matrix. Repeat 200 times.

In [ ]:
Sigma_true = B_true.T @ Sigma_f @ B_true + np.diag(idio**2)

def gmv(S):
    w = np.linalg.solve(S, np.ones(S.shape[0]))
    return w / w.sum()

def true_vol(w):
    return np.sqrt(w @ Sigma_true @ w)

res = []
for rep in range(200):
    Fs = f_mu / 12 + rng.standard_normal((60, 4)) @ np.linalg.cholesky(Sigma_f / 12).T
    Ys = Fs @ B_true + idio / np.sqrt(12) * rng.standard_normal((60, N))
    S_sample = np.cov(Ys, rowvar=False) * 12
    Xs = np.column_stack([np.ones(60), Fs])
    Bs = np.linalg.pinv(Xs) @ Ys
    Es = Ys - Xs @ Bs
    S_factor = Bs[1:].T @ (np.cov(Fs, rowvar=False) * 12) @ Bs[1:] + np.diag(Es.var(axis=0, ddof=5) * 12)
    res.append({"sample": true_vol(gmv(S_sample)), "factor model": true_vol(gmv(S_factor)),
                "err sample": np.linalg.norm(S_sample - Sigma_true), "err factor": np.linalg.norm(S_factor - Sigma_true)})
res = pd.DataFrame(res)
gmv_opt = true_vol(gmv(Sigma_true))
print(f"true minimum-variance volatility: {gmv_opt:.2%}")
print(f"average true vol of GMV built from: sample cov {res['sample'].mean():.2%}, factor model {res['factor model'].mean():.2%}")
print(f"average Frobenius error: sample {res['err sample'].mean():.3f}, factor model {res['err factor'].mean():.3f}")

The minimum-variance portfolio built from the sample covariance has a true volatility of 23.4% on average, more than twice the
optimum of 9.5%: with 50 assets and 60 months the sample matrix is nearly singular and the optimizer chases its noise
(06/04). The factor model gives 10.8%. Its covariance matrix is only modestly closer to the truth entry by entry (Frobenius
error 0.43 against 0.50), but its eigenvalues are far better behaved, which is what the optimizer needs.

### Redundant factors and the pseudoinverse

Add a fifth "factor" that is exactly MKT + SMB (a small-cap index return). $X^\top X$ becomes singular; the pseudoinverse still
works.

In [ ]:
X_red = np.column_stack([X, F[:, 0] + F[:, 1]])
print(f"rank of X_red: {np.linalg.matrix_rank(X_red)} of {X_red.shape[1]} columns,  cond(X'X) = {np.linalg.cond(X_red.T @ X_red):.1e}")
b_red = np.linalg.pinv(X_red) @ y_fund
print("pinv coefficients (alpha, MKT, SMB, HML, MOM, MKT+SMB):", b_red.round(3))
print(f"implied MKT exposure {b_red[1] + b_red[5]:.3f}, SMB exposure {b_red[2] + b_red[5]:.3f}  (4-factor: {b_ff4[1]:.3f}, {b_ff4[2]:.3f})")
print("fitted values identical:", np.allclose(X_red @ b_red, X @ b_ff4))

The pseudoinverse splits the market exposure between MKT (0.51) and the redundant MKT + SMB column (0.52); the implied total
exposures, 1.03 to the market and 0.52 to size, are those of the four-factor regression, and the fitted values are identical.

## 6. Visualizations

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
growth = np.cumprod(1 + F, axis=0)
months = np.arange(1, n + 1) / 12
for k, name in enumerate(factors):
    ax.plot(months, growth[:, k], label=name)
ax.axhline(1, color="black", linewidth=0.8)
ax.set_title("Growth of CHF 1 in each factor (long-short, excess returns), simulated")
ax.set_xlabel("years")
ax.set_ylabel("value")
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(14, 3.8), sharey=False)
for k, ax in enumerate(axes):
    ax.scatter(B_true[k], B_hat[k], s=12, alpha=0.7)
    lo, hi = min(B_true[k].min(), B_hat[k].min()), max(B_true[k].max(), B_hat[k].max())
    ax.plot([lo, hi], [lo, hi], color="grey", linestyle=":")
    ax.set_title(f"{factors[k]} loadings")
    ax.set_xlabel("true")
    ax.grid(True, alpha=0.3)
axes[0].set_ylabel("estimated (10 years)")
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].bar(ret_attr.index, ret_attr.values, color=["tab:blue"] * 4 + ["tab:red"])
axes[0].axhline(0, color="black", linewidth=0.8)
axes[0].yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.1%}"))
axes[0].set_title(f"Return attribution: average excess return {y_fund.mean() * 12:.1%} a year")
axes[1].bar(risk_attr.index, risk_attr.values, color=["tab:blue"] * 4 + ["grey"])
axes[1].axhline(0, color="black", linewidth=0.8)
axes[1].yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0%}"))
axes[1].set_title(f"Risk attribution: shares of variance (vol {np.sqrt(total_var):.1%})")
for ax in axes:
    ax.grid(True, alpha=0.3, axis="y")
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
bins = np.linspace(0.08, max(0.56, res["sample"].max() + 0.01), 97)
ax.hist(res["sample"], bins=bins, histtype="step", linewidth=1.5, label="GMV from sample covariance")
ax.hist(res["factor model"], bins=bins, histtype="step", linewidth=1.5, label="GMV from factor model")
ax.axvline(gmv_opt, color="black", linewidth=0.8, label="true optimum")
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.1%}"))
ax.set_title("True volatility of minimum-variance portfolios, 50 stocks, 60 months (200 repetitions)")
ax.set_xlabel("true annual volatility")
ax.set_ylabel("count")
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()

## 7. Exercises

### Basic

1. A portfolio holds 60% of fund A (1.0, 0.6, 0.4, 0.0) and 40% of fund B (1.1, −0.2, −0.3, 0.4). Compute its loadings.
2. A fund with loadings (0.9, 0.0, 0.6, 0.2) earned 7.5% excess return a year. With premia 6%, 2%, 3%, 5%, how much of it is alpha?
3. A fund reports a CAPM alpha of 2.5% ($t = 2.3$), a four-factor alpha of 0.1% ($t = 0.1$) and an HML loading of 0.5. **Interpret** this for a
   client paying 1.2% in fees.

### Intermediate

4. Recompute the risk attribution of the value fund with the **true** factor covariance and true loadings, and as volatility
   contributions (shares of $\sigma_p$ instead of $\sigma_p^2$ are the same; check that the volatility contributions add up to $\sigma_p$).
5. Using the estimated loadings of the 50 stocks, find the minimum-norm weights $w$ with $\sum w = 1$, market loading 1 and zero
   SMB, HML and MOM loadings (use `np.linalg.pinv` on the constraint matrix). Check its true loadings.

### Advanced

6. Verify the four Penrose conditions for `np.linalg.pinv(X_red)`: $XX^+X = X$, $X^+XX^+ = X^+$, and $XX^+$, $X^+X$ symmetric. Show that among all
   coefficient vectors with the same fitted values, the pseudoinverse one has the smallest norm.
7. Repeat the GMV comparison with $n = 240$ months. Does the advantage of the factor model shrink? Why would it not vanish completely
   if the true model had an omitted industry factor?

## 8. Solutions / Checks

In [ ]:
# 1.
bA, bB = np.array([1.0, 0.6, 0.4, 0.0]), np.array([1.1, -0.2, -0.3, 0.4])
print("1.", dict(zip(factors, (0.6 * bA + 0.4 * bB).round(3))))

# 2.
b2 = np.array([0.9, 0.0, 0.6, 0.2])
print(f"2. factor return {b2 @ f_mu:.2%}, alpha {0.075 - b2 @ f_mu:.2%}")

# 4.
sp2 = beta_fund @ Sigma_f @ beta_fund + 0.03**2
sp = np.sqrt(sp2)
vc = beta_fund * (Sigma_f @ beta_fund) / sp
print(f"4. true vol {sp:.2%}; vol contributions {dict(zip(factors, vc.round(4)))}, idio {0.03**2 / sp:.4f}, "
      f"sum {vc.sum() + 0.03**2 / sp:.4f}")

In [ ]:
# 5.
A = np.vstack([np.ones(N), B_hat])                 # constraints: sum w = 1, B_hat w = target
target = np.array([1.0, 1.0, 0.0, 0.0, 0.0])
w5 = np.linalg.pinv(A) @ target
print(f"5. constraint residual {np.abs(A @ w5 - target).max():.1e}; true loadings {dict(zip(factors, (B_true @ w5).round(3)))}; "
      f"max |w| {np.abs(w5).max():.3f}")

# 6.
Xp = np.linalg.pinv(X_red)
print("6. Penrose:", np.allclose(X_red @ Xp @ X_red, X_red), np.allclose(Xp @ X_red @ Xp, Xp),
      np.allclose((X_red @ Xp).T, X_red @ Xp), np.allclose((Xp @ X_red).T, Xp @ X_red))
null_dir = np.array([0, 1, 1, 0, 0, -1.0])         # X_red @ null_dir = 0
print(f"   X_red @ null direction = {np.abs(X_red @ null_dir).max():.1e}")
for c in [0.0, 0.5, -0.5]:
    bc = b_red + c * null_dir
    print(f"   c = {c:+.1f}: same fit {np.allclose(X_red @ bc, X_red @ b_red)}, norm {np.linalg.norm(bc):.4f}")

In [ ]:
# 7.
res7 = []
for rep in range(200):
    Fs = f_mu / 12 + rng.standard_normal((240, 4)) @ np.linalg.cholesky(Sigma_f / 12).T
    Ys = Fs @ B_true + idio / np.sqrt(12) * rng.standard_normal((240, N))
    Xs = np.column_stack([np.ones(240), Fs])
    Bs = np.linalg.pinv(Xs) @ Ys
    Es = Ys - Xs @ Bs
    S_factor = Bs[1:].T @ (np.cov(Fs, rowvar=False) * 12) @ Bs[1:] + np.diag(Es.var(axis=0, ddof=5) * 12)
    res7.append((true_vol(gmv(np.cov(Ys, rowvar=False) * 12)), true_vol(gmv(S_factor))))
res7 = np.array(res7)
print(f"7. n = 240: sample {res7[:, 0].mean():.2%}, factor {res7[:, 1].mean():.2%}, optimum {gmv_opt:.2%}")

**1.** $(0.6 + 0.44,\ 0.36 - 0.08,\ 0.24 - 0.12,\ 0.16) = (1.04, 0.28, 0.12, 0.16)$.

**2.** Factor return $0.9\cdot 6\% + 0.6\cdot 3\% + 0.2\cdot 5\% = 8.2\%$, so alpha $= 7.5\% - 8.2\% = -0.7\%$.

**3.** The CAPM alpha is explained by the fund's value tilt: once HML (and the other factors) are in the model, nothing
significant is left. The client is paying 1.2% for value exposure that a value index fund or ETF provides for a fraction of
that. Whether the fund adds anything depends on what else it offers (implementation, tax, access), not on skill in stock
selection.

**4.** The volatility contributions add up to $\sigma_p$; market dominates, value is the second-largest source; the shares of $\sigma_p$ equal
the shares of $\sigma_p^2$ because each contribution is divided by the same $\sigma_p$.

**5.** The pseudoinverse finds the smallest-norm weights satisfying the five linear constraints exactly for the **estimated**
loadings. The true loadings are close to the targets but not exactly, because the estimates contain error.

**6.** All four conditions hold. Adding any multiple of the null direction $(0, 1, 1, 0, 0, -1)$ leaves the fit unchanged and increases
the norm: the pseudoinverse picks the coefficient vector orthogonal to the null space.

**7.** With 240 months the sample-covariance portfolio reaches 10.7% and the factor-model portfolio 9.8%, against an optimum of
9.5%: both improve and the gap narrows. With an omitted factor the
factor-model covariance would be biased (residual correlations set to zero), so its error would not go to zero with more
data, while the sample covariance would keep improving: a bias-variance trade-off (08/02).

## 9. Conclusion

- A linear factor model $r - r_f = \alpha + \beta^\top f + \varepsilon$ is estimated for all assets at once with $\hat B = X^+Y$.
- Portfolio exposures are weighted averages of asset exposures; return attribution multiplies exposures by factor
  returns, and alpha is what remains.
- Risk attribution splits $\sigma_p^2 = \beta^\top\Sigma_f\beta + \sigma_\varepsilon^2$ with Euler contributions.
- A CAPM alpha can be a factor premium; compare against the multi-factor alpha before paying for skill.
- The factor-model covariance $B^\top\Sigma_f B + D$ needs far fewer parameters and produces better minimum-variance portfolios when data
  are short.
- With redundant factors the pseudoinverse gives the minimum-norm coefficients; the fit is unique, the split is not.

Next: [07/04 PCA of returns: statistical factors](./04_pca_of_returns_statistical_factors.ipynb).